# Assignment3: Snakemake Pipeline (Alzheimer CCF Annotation)

이 노트북은 기존 `CCF_annotation.ipynb` 분석 흐름을 **재현 가능한 파이프라인**으로 바꾼 버전입니다.

## 목표
1. 수작업으로 실행하던 분석 단계를 rule로 분리
2. 입력/출력 의존성을 명확히 정의
3. `snakemake --cores N` 한 번으로 전체 실행 가능하게 구성

## 분석 단계 매핑 (Notebook -> Snakemake)
- Step A: 입력 파일 검증 -> `rule check_inputs`
- Step B: micron->mosaic 및 CCF 좌표 계산 -> `rule transform_coords`
- Step C: CCF region label 부여 -> `rule assign_ccf_labels`
- Step D: QC summary/plot 생성 -> `rule make_qc`
- Step E: 최종 결과 저장(h5ad/csv) -> `rule export_outputs`

In [1]:
# 1) Workspace and pipeline directories
from pathlib import Path

base_dir = Path('/home/Data_Drive_8TB/kykim/0. Jupyter/Alzheimer/snakemake_ccf_pipeline')
workflow_dir = base_dir / 'workflow'
scripts_dir = workflow_dir / 'scripts'
results_dir = base_dir / 'results'
logs_dir = base_dir / 'logs'
config_dir = base_dir / 'config'

for d in [base_dir, workflow_dir, scripts_dir, results_dir, logs_dir, config_dir]:
    d.mkdir(parents=True, exist_ok=True)

print('Pipeline root:', base_dir)
print('Created directories:')
for d in [workflow_dir, scripts_dir, results_dir, logs_dir, config_dir]:
    print(' -', d)

Pipeline root: /home/Data_Drive_8TB/kykim/0. Jupyter/Alzheimer/snakemake_ccf_pipeline
Created directories:
 - /home/Data_Drive_8TB/kykim/0. Jupyter/Alzheimer/snakemake_ccf_pipeline/workflow
 - /home/Data_Drive_8TB/kykim/0. Jupyter/Alzheimer/snakemake_ccf_pipeline/workflow/scripts
 - /home/Data_Drive_8TB/kykim/0. Jupyter/Alzheimer/snakemake_ccf_pipeline/results
 - /home/Data_Drive_8TB/kykim/0. Jupyter/Alzheimer/snakemake_ccf_pipeline/logs
 - /home/Data_Drive_8TB/kykim/0. Jupyter/Alzheimer/snakemake_ccf_pipeline/config


In [2]:
# 2) Write config.yaml (input paths + sample mappings)
import yaml

config = {
    'data_dir': '/home/Data_Drive_8TB_2/kykim/potrai/DAPI_downsampling',
    'adata_path': '/home/Data_Drive_8TB_2/kykim/potrai/MERSCOPE/after_label_transfer_manual.h5ad',
    'atlas_annot_path': '/home/Data_Drive_8TB_2/kykim/potrai/DAPI_downsampling/annotation_25.nrrd',
    'atlas_template_path': '/home/Data_Drive_8TB_2/kykim/potrai/DAPI_downsampling/average_template_25.nrrd',
    'structure_tree_path': '/home/Data_Drive_8TB_2/kykim/potrai/DAPI_downsampling/structure_tree_safe_2017.csv',
    'allow_provisional_auto_ccf': False,
    'provisional_z_search_step': 5,
    'sample_to_micron2px_csv': {
        'WT1': 'WT-5xFAD10518pHip300GP_region_0_micron_to_mosaic_pixel_transform.csv',
        '5xFAD1': 'WT-5xFAD10518pHip300GP_region_1_micron_to_mosaic_pixel_transform.csv',
        'WT3': '202209041036_5xFAD-TREM2-AnteHip-Region_1_micron_to_mosaic_pixel_transform.csv',
        '5xFAD3': '202209041036_5xFAD-TREM2-AnteHip-Region_3_micron_to_mosaic_pixel_transform.csv'
    }
}

config_path = config_dir / 'config.yaml'
with open(config_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(config, f, sort_keys=False, allow_unicode=False)

print('Saved:', config_path)

Saved: /home/Data_Drive_8TB/kykim/0. Jupyter/Alzheimer/snakemake_ccf_pipeline/config/config.yaml


In [ ]:
# 3) Create step scripts used by Snakemake
from textwrap import dedent

step01 = dedent('''
import os
import yaml

with open(snakemake.input.config, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

required = [
    cfg['adata_path'],
    cfg['atlas_annot_path'],
    cfg['atlas_template_path'],
    cfg['structure_tree_path'],
]

missing = [p for p in required if not os.path.exists(p)]
if missing:
    raise FileNotFoundError('Missing required files: ' + ', '.join(missing))

with open(snakemake.output.done, 'w', encoding='utf-8') as f:
    f.write('OK\\n')
''').strip()

step02 = dedent('''
import scanpy as sc
import pandas as pd
import numpy as np

adata = sc.read_h5ad(snakemake.params.adata_path)

# Placeholder: keep structure and pass-through for pipeline skeleton
# Replace this with full transform logic from CCF_annotation notebook cell 4/9 if needed.
numeric_cols = ['x_mosaic_px', 'y_mosaic_px', 'x_ccf_vox', 'y_ccf_vox', 'z_ccf_vox']
for col in numeric_cols:
    if col not in adata.obs.columns:
        adata.obs[col] = np.nan

if 'ccf_reg_mode' not in adata.obs.columns:
    adata.obs['ccf_reg_mode'] = 'none'

adata.write_h5ad(snakemake.output.h5ad)
''').strip()

step03 = dedent('''
import numpy as np
import pandas as pd
import scanpy as sc

adata = sc.read_h5ad(snakemake.input.h5ad)

# Placeholder label assignment (replace with atlas-based assignment)
if 'CCF_acronym' not in adata.obs.columns:
    adata.obs['CCF_acronym'] = 'Not_mapped'
if 'CCF_name' not in adata.obs.columns:
    adata.obs['CCF_name'] = 'Not_mapped'
if 'ccf_id' not in adata.obs.columns:
    adata.obs['ccf_id'] = -1

adata.write_h5ad(snakemake.output.h5ad)
''').strip()

step04 = dedent('''
import scanpy as sc
import matplotlib.pyplot as plt

adata = sc.read_h5ad(snakemake.input.h5ad)

fig, ax = plt.subplots(figsize=(6, 4))
counts = adata.obs['CCF_acronym'].value_counts().head(20) if 'CCF_acronym' in adata.obs else None

if counts is not None and len(counts) > 0:
    counts.plot(kind='bar', ax=ax)
    ax.set_title('Top 20 CCF labels')
    ax.set_xlabel('CCF_acronym')
    ax.set_ylabel('count')
    plt.xticks(rotation=70)
else:
    ax.text(0.5, 0.5, 'No CCF labels available', ha='center', va='center')
    ax.set_axis_off()

plt.tight_layout()
plt.savefig(snakemake.output.png, dpi=200)
''').strip()

step05 = dedent('''
import scanpy as sc

adata = sc.read_h5ad(snakemake.input.h5ad)
adata.write_h5ad(snakemake.output.h5ad)
adata.obs.to_csv(snakemake.output.csv)
''').strip()

scripts = {
    '01_check_inputs.py': step01,
    '02_transform_coords.py': step02,
    '03_assign_ccf_labels.py': step03,
    '04_make_qc.py': step04,
    '05_export_outputs.py': step05,
}

for name, content in scripts.items():
    p = scripts_dir / name
    p.write_text(content + '\n', encoding='utf-8')
    print('Wrote', p)

Wrote /home/Data_Drive_8TB/kykim/0. Jupyter/Alzheimer/snakemake_ccf_pipeline/workflow/scripts/01_check_inputs.py
Wrote /home/Data_Drive_8TB/kykim/0. Jupyter/Alzheimer/snakemake_ccf_pipeline/workflow/scripts/02_transform_coords.py
Wrote /home/Data_Drive_8TB/kykim/0. Jupyter/Alzheimer/snakemake_ccf_pipeline/workflow/scripts/03_assign_ccf_labels.py
Wrote /home/Data_Drive_8TB/kykim/0. Jupyter/Alzheimer/snakemake_ccf_pipeline/workflow/scripts/04_make_qc.py
Wrote /home/Data_Drive_8TB/kykim/0. Jupyter/Alzheimer/snakemake_ccf_pipeline/workflow/scripts/05_export_outputs.py


In [4]:
# 4) Write Snakefile
snakefile_text = dedent('''
configfile: 'config/config.yaml'

rule all:
    input:
        'results/final/after_label_transfer_ccf_annotated.h5ad',
        'results/final/after_label_transfer_ccf_obs.csv',
        'results/qc/ccf_top20.png'

rule check_inputs:
    input:
        config='config/config.yaml'
    output:
        done='results/intermediate/check_inputs.done'
    log:
        'logs/01_check_inputs.log'
    script:
        'workflow/scripts/01_check_inputs.py'

rule transform_coords:
    input:
        done='results/intermediate/check_inputs.done'
    output:
        h5ad='results/intermediate/02_transformed.h5ad'
    params:
        adata_path=config['adata_path']
    log:
        'logs/02_transform_coords.log'
    script:
        'workflow/scripts/02_transform_coords.py'

rule assign_ccf_labels:
    input:
        h5ad='results/intermediate/02_transformed.h5ad'
    output:
        h5ad='results/intermediate/03_labeled.h5ad'
    log:
        'logs/03_assign_ccf_labels.log'
    script:
        'workflow/scripts/03_assign_ccf_labels.py'

rule make_qc:
    input:
        h5ad='results/intermediate/03_labeled.h5ad'
    output:
        png='results/qc/ccf_top20.png'
    log:
        'logs/04_make_qc.log'
    script:
        'workflow/scripts/04_make_qc.py'

rule export_outputs:
    input:
        h5ad='results/intermediate/03_labeled.h5ad'
    output:
        h5ad='results/final/after_label_transfer_ccf_annotated.h5ad',
        csv='results/final/after_label_transfer_ccf_obs.csv'
    log:
        'logs/05_export_outputs.log'
    script:
        'workflow/scripts/05_export_outputs.py'
''').strip() + '\n'

snakefile_path = base_dir / 'Snakefile'
snakefile_path.write_text(snakefile_text, encoding='utf-8')
print('Saved:', snakefile_path)

Saved: /home/Data_Drive_8TB/kykim/0. Jupyter/Alzheimer/snakemake_ccf_pipeline/Snakefile


In [5]:
# 5) Create minimal conda environment yaml for Snakemake execution
env_yaml = dedent('''
name: ccf_snakemake
channels:
  - conda-forge
  - bioconda
dependencies:
  - python=3.11
  - snakemake-minimal
  - numpy
  - pandas
  - scanpy
  - matplotlib
  - pyyaml
  - pynrrd
''').strip() + '\n'

env_path = base_dir / 'environment.yml'
env_path.write_text(env_yaml, encoding='utf-8')
print('Saved:', env_path)

Saved: /home/Data_Drive_8TB/kykim/0. Jupyter/Alzheimer/snakemake_ccf_pipeline/environment.yml


In [6]:
# 6) Command cheatsheet for Assignment3 report
commands = [
    f'cd "{base_dir}"',
    'snakemake -n -p',
    'snakemake --cores 4 -p',
    'snakemake --dag | dot -Tpng > dag.png',
    'snakemake --summary'
]

print('\n'.join(commands))

cd "/home/Data_Drive_8TB/kykim/0. Jupyter/Alzheimer/snakemake_ccf_pipeline"
snakemake -n -p
snakemake --cores 4 -p
snakemake --dag | dot -Tpng > dag.png
snakemake --summary


## 과제 제출 시 포인트 (Assignment3)
- 왜 Snakemake가 필요한지: 재현성, 자동화, 의존성 관리
- 각 rule의 입력/출력/로그를 명시해 추적 가능성 확보
- `-n` dry-run 결과와 DAG 이미지를 함께 제출하면 설계 의도가 잘 보임
- 실제 분석 로직은 `workflow/scripts/02_transform_coords.py`, `03_assign_ccf_labels.py`에 확장

현재 버전은 **과제용 파이프라인 뼈대**이며, CCF 매핑 핵심 로직을 단계 스크립트에 옮겨 넣으면 바로 실사용 가능합니다.